# 510 - Response timing of the functional groups

**The question 140 cannot answer.** Everything upstream is time-normalised: every
trial is warped to 300 bins, 0% is stimulus onset, 50% is the GO cue. A latency in
that space is a percentage of a trial whose real duration was discarded. The
clustering (02) and the pooling roles (04) both live there.

This notebook uses the **real-time, GO-aligned** cubes from 150 to ask: in seconds
after the GO cue, in what order do those groups respond, and does the order survive
the move from one condition to another?

> **NOTES:** *t = 0 is the GO CUE, i.e. stimulus offset. There is no speech-onset
> event anywhere in this dataset, so 'response portion' means 'after the stimulus
> ended', never 'after the patient began speaking'.*


## 0 - Setup

`lf_rt` is the only place that knows how to rebuild the two axes, which are **not**
stored in the `.npy`. If the RT window in 150 ever changes, it changes there.


In [ ]:
import sys, json
from pathlib import Path
import numpy as np, pandas as pd, matplotlib.pyplot as plt
sys.path.insert(0, 'functions')
import lf_rt as R

print('window', R.RT_WINDOW, '| grid step', round(float(np.diff(R.GRID)[0]), 3), 's')
print('grid', R.GRID[0], '->', round(float(R.GRID[-1]), 2), 'n =', len(R.GRID))


## 1 - What one cube looks like

Sanity first: reconstruct the axes for a single electrode and check that the picture
matches the ERSP figure the pipeline drew for it. If this looks wrong, nothing below
is worth reading.


In [ ]:
sys.path.insert(0, str(Path.cwd().parent / '01_FBM_Analysis'))
from functions import config as cfg
import os

RT_ROOT = os.path.join(cfg.outputs_root, '05_ERSP_LM_RAWONLY_RealTime')
recs = list(R.iter_cubes(RT_ROOT))
print(f'{len(recs)} cubes in the tree')

rec = recs[0]
A = np.load(rec['path'])
x, f = R.time_axis(A.shape[1]), R.freq_axis(A.shape[0])
print(rec['patient_id'], rec['condition'], rec['electrode'], A.shape)

fig, (a1, a2) = plt.subplots(2, 1, figsize=(9, 6), sharex=True,
                             gridspec_kw=dict(height_ratios=[2, 1]))
a1.imshow(A, aspect='auto', origin='lower', cmap='bwr', vmin=-6, vmax=6,
          extent=[x[0], x[-1], f[0], f[-1]])
a1.set_ylim(0, 400); a1.set_ylabel('Hz'); a1.axvline(0, color='k', lw=1)
a1.set_title(f"{rec['patient_id']} {rec['condition']} {rec['electrode']}  "
             f"(t=0 is the GO cue)", loc='left')
a2.plot(R.GRID, R.to_grid(R.band_trace(A)), lw=1.4)
a2.axhline(0, color='0.8', lw=.8); a2.axvline(0, color='k', lw=1)
a2.set_xlabel('s after GO'); a2.set_ylabel('70-150 Hz (dB)')
plt.tight_layout(); plt.show()


## 2 - The timing table

One row per electrode x condition. Built by `build_timing_table.py` so it can run
without a kernel; this cell just loads what that produced.

> **NOTES:** *The cubes are TRIAL-AVERAGED. Per-trial data is not persisted anywhere
> in the 05 tree, so nothing here can correlate one trial's response time against one
> trial's neural timing. Every comparison below is across ELECTRODES.*


In [ ]:
T = pd.read_csv('outputs/timing/timing_table.csv')
meta = json.loads(Path('outputs/timing/meta.json').read_text())
print(f"{len(T)} rows | {T.patient_id.nunique()} patients")
print('per condition   :', meta['per_condition'])
print('label coverage  :', meta['label_coverage'])
T.head(3)


### 2a - How many electrodes actually have an onset?

`onset_lat` requires +1 dB sustained for 100 ms. Plenty of electrodes never do that,
and they drop out of the onset rows rather than being scored as arbitrarily late.
Worth knowing before reading any onset comparison.


In [ ]:
for c in ['audio', 'picture', 'reading']:
    d = T[T.condition == c]
    print(f"{c:9s} n={len(d):5d}  peak_lat {d.peak_lat.notna().sum():5d}"
          f"  onset_lat {d.onset_lat.notna().sum():5d}"
          f"  ({100*d.onset_lat.notna().mean():.0f}%)")


## 3 - Do the groups differ in timing?

**Kruskal-Wallis is the wrong test here and it is instructive to see why.** Electrodes
within a patient share a brain, a reference, a montage and a response speed, so they
are nowhere near independent. Pooled across electrodes, KW returns p < 1e-10 for every
group and every condition - including combinations that the honest test calls null.

The headline test shuffles group labels **within each patient**, so it asks: given
this patient's own contacts, does group membership still order them in time?


In [ ]:
S = pd.read_csv('outputs/tables/group_timing_stats.csv')
piv = S.pivot_table(index=['group', 'feature'], columns='condition',
                    values='perm_p_within_patient')
kw = S.pivot_table(index=['group', 'feature'], columns='condition',
                   values='kruskal_p')
print('within-patient permutation p'); display(piv.round(4))
print('Kruskal-Wallis p, for contrast'); display(kw.map(lambda v: f'{v:.1e}'))


## 4 - The ordering itself

`RT-T2` is the figure this folder exists for: each group's median latency with a
bootstrap CI, per condition, groups sorted by median. Read the ORDER, not the absolute
values - the 5 s cap and the trial-averaging both compress the tail.


In [ ]:
from IPython.display import Image, display as _d
for g in ['hier_concat_hg', 'kmeans_concat_hg', 'cnmf_lead']:
    p = Path(f'outputs/figures/RT-T2_{g}_peak_lat.png')
    if p.exists():
        print(g); _d(Image(str(p)))


## 5 - Does an electrode keep its timing across conditions?

If a contact is early in audio, is it early in reading? A high correlation means the
timing is a property of the SITE; a low one means it is a property of the TASK. Both
are informative and they point at different papers.


In [ ]:
C = pd.read_csv('outputs/tables/cross_condition_consistency.csv')
display(C)
p = Path('outputs/figures/RT-T3_consistency_peak_lat.png')
if p.exists(): _d(Image(str(p)))


## 6 - The graded view

The convex-NMF decomposition finds that ~66% of electrodes have **no majority
component**. Forcing an argmax label throws that away. Correlating the component
WEIGHT against latency keeps it: an electrode that is half component 2 and half
component 5 contributes to both.


In [ ]:
L = pd.read_csv('outputs/tables/loading_vs_timing.csv')
top = L.reindex(L.spearman_rho.abs().sort_values(ascending=False).index).head(12)
display(top)
p = Path('outputs/figures/RT-T4_loading_vs_peak_lat.png')
if p.exists(): _d(Image(str(p)))


## 7 - The confound that would explain it all away

If electrode latency simply tracks how long that PATIENT took to respond, then any
group difference could be cohort composition rather than physiology - a group that
happens to contain more contacts from slow responders would look late. The
within-patient permutation in section 3 already controls for this; this figure shows
the raw relationship so the size of the effect is visible rather than assumed.


In [ ]:
F = pd.read_csv('outputs/tables/patient_confound.csv')
display(F)
p = Path('outputs/figures/RT-T5_patient_confound_peak_lat.png')
if p.exists(): _d(Image(str(p)))


## 8 - What is still open

> **NOTES:**
> - *Per-trial HG is not saved by 150. Until it is, the response-time question is
>   answered across electrodes and never within a trial - which is the version of the
>   question most reviewers will actually ask.*
> - *`RT_MAX_POST_S = 5.0` is uniform across patients, but response durations are not.
>   EL046's reading median is 7.9 s, so that block loses most of its trials while
>   others lose none.*
> - *PAT_6953 (151 electrodes) and EL046 (80) have no fsaverage reconstruction, so they
>   are in every timing comparison but absent from every anatomical one.*
> - *Four patients have partial coverage in the 05 tree: EL043 no picture, EL044 audio
>   only, PAT_3301 picture only, PAT_3965 reading only.*
